# FINC3014 — Analysis notebook

This notebook turns your **IBKR trade log** into the performance and
execution-cost numbers your README reports. It is graded on **reproducibility**
(criterion B1: it must run top-to-bottom from a fresh kernel) and on the
**required outputs** (B3–B6).

**To use it on your real data:** replace `trades.csv` with your IBKR trade-log
export (keep the column names), and replace `prices.csv` with daily closes for
your traded symbols **and** your benchmark. The notebook ships with *sample*
data so it runs out of the box — delete the sample once your data is in.

> Run order: `Kernel → Restart & Run All`. If that errors, you lose B1.

## 0. Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

START_CAPITAL = 1_000_000.0   # USD simulated capital
RF_ANNUAL     = 0.045         # risk-free rate (annual) -- update to the window's T-bill rate
BENCH         = 'SPY'         # benchmark column in prices.csv -- change to match YOUR asset class
TRADING_DAYS  = 252

## 1. Load the data

`trades.csv` columns (this is the schema your IBKR export should match):
`datetime, symbol, action (BUY/SELL), quantity, price, commission, mid_at_order`.

- `mid_at_order` is the mid-quote at the moment you submitted the order.
  **Record it when you trade** (screenshot the order book) -- you need it for the
  effective-spread cost metric, and the platform will not give it to you later.

In [ ]:
prices = pd.read_csv('prices.csv', parse_dates=['date']).set_index('date')
trades = pd.read_csv('trades.csv', parse_dates=['datetime'])
trades['date'] = trades['datetime'].dt.normalize()

# Signed quantity (+buy / -sell) and the cash impact of each fill
trades['signed_qty'] = np.where(trades['action'].eq('BUY'), trades['quantity'], -trades['quantity'])
trades['cash_flow']  = -trades['signed_qty'] * trades['price'] - trades['commission']

sym_cols = [c for c in prices.columns if c != BENCH]
print(f'{len(trades)} trades across {trades.symbol.nunique()} symbols; benchmark = {BENCH}')
trades

## 2. Reconstruct the daily equity curve

Method: cumulative positions per symbol (forward-filled across all dates),
marked to market on each day's close, plus a running cash balance.

$$\text{Equity}_t = \text{Cash}_t + \sum_{s}\big(\text{Position}_{s,t}\times P_{s,t}\big)$$

In [ ]:
pos = pd.DataFrame(0.0, index=prices.index, columns=sym_cols)
for s in sym_cols:
    daily = trades[trades.symbol.eq(s)].groupby('date')['signed_qty'].sum()
    pos[s] = daily.reindex(prices.index, fill_value=0).cumsum()

cash_flow_by_date = trades.groupby('date')['cash_flow'].sum().reindex(prices.index, fill_value=0)
cash      = START_CAPITAL + cash_flow_by_date.cumsum()
holdings  = (pos * prices[sym_cols]).sum(axis=1)
equity    = cash + holdings

port_ret  = equity.pct_change().fillna(0.0)
bench_ret = prices[BENCH].pct_change().fillna(0.0)
equity.plot(title='Portfolio equity (USD)', figsize=(8,3), grid=True); plt.show()

## 3. Performance metrics  *(criterion B3 — ≥2 metrics, with formulas)*

**Total return:** $\;R = \dfrac{V_T}{V_0}-1$

**Annualised Sharpe ratio:** $\;\text{SR}=\sqrt{252}\,\dfrac{\overline{r-r_f}}{\sigma_r}$
  — note we subtract the (daily) risk-free rate. Forgetting $r_f$ is a conceptual error.

**Maximum drawdown:** $\;\text{MDD}=\min_t\!\left(\dfrac{V_t}{\max_{u\le t}V_u}-1\right)$

In [ ]:
def sharpe(r, rf_annual=RF_ANNUAL, periods=TRADING_DAYS):
    sd = r.std(ddof=1)
    if sd == 0:
        return np.nan
    excess = r - rf_annual/periods
    return np.sqrt(periods) * excess.mean() / sd

def max_drawdown(curve):
    return (curve / curve.cummax() - 1.0).min()

summary = pd.DataFrame({
    'Portfolio': [equity.iloc[-1]/equity.iloc[0]-1, sharpe(port_ret), max_drawdown(equity)],
    'Benchmark': [prices[BENCH].iloc[-1]/prices[BENCH].iloc[0]-1, sharpe(bench_ret), max_drawdown(prices[BENCH])],
}, index=['Total return', 'Sharpe (ann.)', 'Max drawdown'])
summary

## 4. Benchmark comparison chart  *(criterion B5)*

One chart, both lines, legend and axis labels. Paste this figure into your README.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
((1+port_ret).cumprod()-1).plot(ax=ax, label='Portfolio')
((1+bench_ret).cumprod()-1).plot(ax=ax, label=f'Benchmark ({BENCH})')
ax.set_title('Cumulative return: portfolio vs benchmark')
ax.set_xlabel('Date'); ax.set_ylabel('Cumulative return')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

## 5. Execution-cost metrics  *(criterion B4 — ≥2 metrics, with formulas)*

**Effective half-spread** for a trade with direction $d$ (+1 buy, $-1$ sell):
$$\text{eff half-spread}=d\,(P_{\text{fill}}-M_{\text{mid}}),\qquad
\text{in bps}=10^4\times\dfrac{d\,(P_{\text{fill}}-M_{\text{mid}})}{M_{\text{mid}}}$$

**Commission cost** in bps of notional: $\;10^4\times\dfrac{\text{commission}}{P_{\text{fill}}\times\text{qty}}$

_Extensions you may add (and earn the same mark):_ realised spread (needs the
mid a few minutes after the trade), VWAP slippage, implementation shortfall.

In [ ]:
t = trades.copy()
d = np.where(t.action.eq('BUY'), 1, -1)
t['eff_half_spread']     = d * (t['price'] - t['mid_at_order'])
t['eff_half_spread_bps'] = 1e4 * t['eff_half_spread'] / t['mid_at_order']
t['commission_bps']      = 1e4 * t['commission'] / (t['price'] * t['quantity'])

print(f"Mean effective half-spread: {t['eff_half_spread_bps'].mean():.2f} bps")
print(f"Mean commission cost:       {t['commission_bps'].mean():.2f} bps")
worst = t.loc[t['eff_half_spread_bps'].idxmax()]
print(f"Highest-cost trade (eff. spread): {worst['symbol']} {worst['action']} "
      f"@ {worst['eff_half_spread_bps']:.1f} bps")
t[['symbol','action','price','mid_at_order','eff_half_spread_bps','commission_bps']].round(3)

## 6. Performance attribution  *(criteria C7, C8)*

Decompose total P&L into buckets (here: by symbol) and check the buckets
**reconcile to the total**. Then read off the largest contributor / detractor
and explain them in the README by linking to your journal decisions.

P&L per symbol = (sum of that symbol's cash flows) + (final mark of any residual position).

In [ ]:
final_px = prices.iloc[-1]
pnl = {}
for s in sym_cols:
    cf = trades[trades.symbol.eq(s)]['cash_flow'].sum()
    residual = pos[s].iloc[-1] * final_px[s]
    pnl[s] = cf + residual
pnl = pd.Series(pnl).sort_values()

print(pnl.round(2).to_string())
print(f'\nSum of buckets : {pnl.sum():,.2f}')
print(f'Equity change  : {equity.iloc[-1]-equity.iloc[0]:,.2f}   <- must match (reconciliation)')
print(f'Largest contributor: {pnl.idxmax()}   Largest detractor: {pnl.idxmin()}')
pnl.plot(kind='barh', title='P&L attribution by symbol (USD)', grid=True); plt.show()

## 7. Your turn — extend this notebook

The cells above already satisfy B3, B4, B5 and the C7/C8 inputs on the *sample*
data. To finish:

- [ ] Replace `trades.csv` (your IBKR export) and `prices.csv` (your symbols + benchmark).
- [ ] Set `BENCH` and `RF_ANNUAL` correctly for your portfolio and window.
- [ ] (Optional) add a third performance metric (e.g. Sortino, information ratio).
- [ ] (Optional) add realised spread or implementation shortfall to §5.
- [ ] Copy the chart from §4 and the tables from §3/§5/§6 into the README,
      and make sure the numbers **match** (criterion B6, C12).
- [ ] `Kernel → Restart & Run All` one last time before you commit (criterion B1).